In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time
import os # Tambahan: Untuk mengecek keberadaan file

def scrape_food_data(search_query, max_pages=10):
    """
    Fungsi untuk melakukan scraping data makanan dengan fitur PAGINASI.
    Mencari hingga max_pages untuk setiap kata kunci agar mendapat ribuan data.
    """
    # Mengganti spasi dengan format URL (misal: "nasi goreng" menjadi "nasi+goreng")
    query_formatted = search_query.replace(' ', '+')

    food_list = []

    # LOOPING HALAMAN (PAGINASI)
    for page in range(max_pages):
        # Menambahkan parameter &pg={page} ke URL untuk membuka halaman berikutnya
        url = f"https://www.fatsecret.co.id/kalori-gizi/search?q={query_formatted}&pg={page}"

        # Menambahkan 'Headers' agar kita terlihat seperti browser asli, bukan robot/bot
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
        }

        print(f"  -> Mengambil Halaman {page + 1} untuk: '{search_query}'...")

        try:
            # Mengirim permintaan ke website
            response = requests.get(url, headers=headers)
            response.raise_for_status() # Mengecek apakah website berhasil diakses (Status 200)
        except requests.exceptions.RequestException as e:
            print(f"Gagal mengakses halaman: {e}")
            break # Berhenti jika gagal akses halaman, lanjut ke kata kunci lain

        # Mengubah teks HTML menjadi objek BeautifulSoup agar mudah diekstrak
        soup = BeautifulSoup(response.text, 'html.parser')

        # Mencari semua wadah/kotak yang berisi informasi makanan
        items = soup.find_all('td', class_='borderBottom')

        if not items:
            print(f"  -> Tidak ada data lagi di halaman {page + 1}. Beralih ke kata kunci lain.")
            break # Keluar dari loop halaman jika data di halaman ini kosong

        for item in items:
            try:
                # 1. Mengambil Nama Makanan (biasanya ada di dalam tag <a> dengan class 'prominent')
                name_tag = item.find('a', class_='prominent')
                if not name_tag:
                    continue
                food_name = name_tag.text.strip()

                # 2. Mengambil Ringkasan Gizi (biasanya ada di div kecil dengan class 'smallText')
                # Contoh teks: "100 g - Kalori: 105kkal | Lemak: 0,39g | Karb: 26,98g | Prot: 1,29g"
                nutrition_tag = item.find('div', class_='smallText')
                if not nutrition_tag:
                    continue
                nutrition_text = nutrition_tag.text.strip()

                # FITUR BARU: Mengekstrak Ukuran Porsi (biasanya berada sebelum tanda '-')
                ukuran_porsi = "1 Porsi (Default)" # Nilai default jika formatnya berbeda
                if '-' in nutrition_text:
                    ukuran_porsi = nutrition_text.split('-')[0].strip()

                # 3. Mengekstrak angka menggunakan Regex (Regular Expression)
                # Regex ini mencari pola kata, diikuti titik dua, spasi, angka, dan satuan
                calories = re.search(r'Kalori:\s*([\d,]+)kkal', nutrition_text)
                fat = re.search(r'Lemak:\s*([\d,]+)g', nutrition_text)
                carbs = re.search(r'Karb:\s*([\d,]+)g', nutrition_text)
                protein = re.search(r'Prot:\s*([\d,]+)g', nutrition_text)

                # Membersihkan koma menjadi titik agar bisa dibaca sebagai angka (float) di Python
                cal_val = float(calories.group(1).replace(',', '.')) if calories else 0.0
                fat_val = float(fat.group(1).replace(',', '.')) if fat else 0.0
                carbs_val = float(carbs.group(1).replace(',', '.')) if carbs else 0.0
                protein_val = float(protein.group(1).replace(',', '.')) if protein else 0.0

                # --- FITUR STANDARISASI PORSI (NORMALISASI KE 100g / 100ml) ---
                # Mencari besaran angka gram atau ml menggunakan regex
                match_berat = re.search(r'(\d+(?:[\.,]\d+)?)\s*(g|ml)', ukuran_porsi.lower())

                if match_berat:
                    berat_asli = float(match_berat.group(1).replace(',', '.'))
                    satuan = match_berat.group(2) # akan bernilai 'g' atau 'ml'

                    if berat_asli > 0 and berat_asli != 100:
                        # Rumus Matematika: Konversi Gizi ke ukuran 100g/100ml
                        faktor = 100.0 / berat_asli
                        cal_val = round(cal_val * faktor, 2)
                        fat_val = round(fat_val * faktor, 2)
                        carbs_val = round(carbs_val * faktor, 2)
                        protein_val = round(protein_val * faktor, 2)

                    # Seragamkan kolom ukuran porsi menjadi persis "100 g" atau "100 ml"
                    ukuran_porsi = f"100 {satuan}"
                else:
                    # Jika takaran tidak menggunakan gram/ml (misal: "1 mangkok", "1 piring"),
                    # kita LEWATI (skip) makanan ini agar dataset tetap murni dan merata.
                    continue
                # --------------------------------------------------------------

                # Menyimpan data makanan ke dalam daftar beserta fitur baru
                food_list.append({
                    'Nama Makanan': food_name,
                    'Ukuran Porsi': ukuran_porsi,
                    'Kalori (kkal)': cal_val,
                    'Karbohidrat (g)': carbs_val,
                    'Lemak (g)': fat_val,
                    'Protein (g)': protein_val
                })

            except Exception as e:
                # Jika ada struktur yang tidak sesuai, lewati saja dan lanjut ke makanan berikutnya
                print(f"Melewati satu item karena error: {e}")
                continue

        # JEDA WAKTU ANTAR HALAMAN (Sangat penting agar IP tidak diblokir server)
        time.sleep(2)

    return food_list

# === BAGIAN UTAMA PROGRAM ===
if __name__ == "__main__":
    # NAMA FILE UNTUK INPUT DAN OUTPUT
    file_input = "daftar_makanan.txt"
    nama_file_output = "dataset_gizi_polaku.csv"

    daftar_pencarian = []

    # 1. Cek apakah ada file daftar_makanan.txt berisi ribuan makanan
    if os.path.exists(file_input):
        print(f"Membaca daftar makanan dari file {file_input}...")
        with open(file_input, 'r', encoding='utf-8') as f:
            # Membaca setiap baris, membersihkan spasi, dan mengabaikan baris kosong
            daftar_pencarian = [line.strip() for line in f if line.strip()]
    else:
        print(f"File {file_input} tidak ditemukan. Membuat daftar pencarian OTOMATIS secara massal...")
        # STRATEGI PAGINASI MASSAL:
        # Kita menggunakan kata dasar yang luas. Jika tiap kata menghasilkan 10 halaman
        # dan tiap halaman ada 10 item, kita bisa mendapatkan sekitar 4.200 data makanan!
        daftar_pencarian = [
            "Nasi", "Mie", "Ayam", "Sapi", "Daging", "Ikan", "Udang", "Cumi", "Kerang",
            "Tahu", "Tempe", "Telur", "Sayur", "Bayam", "Kangkung", "Brokoli", "Wortel",
            "Buah", "Pisang", "Apel", "Jeruk", "Susu", "Keju", "Yogurt", "Roti", "Kue",
            "Kacang", "Sambal", "Kerupuk", "Gorengan", "Soto", "Sate", "Bakso", "Bubur",
            "Bakar", "Rebus", "Tumis", "Panggang", "Kuah", "Goreng", "Kopi", "Teh"
        ]

    total_item = len(daftar_pencarian)
    print(f"Total kata kunci yang akan dicari: {total_item} kata dasar.\n")

    # 2. Siapkan file CSV (buat header-nya jika file belum ada, tambahkan Ukuran Porsi)
    if not os.path.exists(nama_file_output):
        df_kosong = pd.DataFrame(columns=['Nama Makanan', 'Ukuran Porsi', 'Kalori (kkal)', 'Karbohidrat (g)', 'Lemak (g)', 'Protein (g)'])
        df_kosong.to_csv(nama_file_output, index=False, encoding='utf-8')

    # 3. Looping untuk mencari setiap makanan
    for i, kata_kunci in enumerate(daftar_pencarian):
        # Indikator progres di layar komputer
        print(f"\n[{i+1}/{total_item}] Memulai pencarian kata kunci: {kata_kunci}")

        # FITUR BARU: Menambahkan argumen max_pages=10
        hasil = scrape_food_data(kata_kunci, max_pages=10)

        if hasil:
            # Simpan data langsung ke CSV (mode='a' berarti append / menambahkan ke baris paling bawah)
            df_sementara = pd.DataFrame(hasil)
            df_sementara.to_csv(nama_file_output, mode='a', header=False, index=False, encoding='utf-8')
            print(f"✅ Berhasil menyimpan total {len(hasil)} item untuk kata kunci '{kata_kunci}'")

    print(f"\nSelesai! Seluruh data telah berhasil dikumpulkan di file: {nama_file_output}")

File daftar_makanan.txt tidak ditemukan. Membuat daftar pencarian OTOMATIS secara massal...
Total kata kunci yang akan dicari: 42 kata dasar.


[1/42] Memulai pencarian kata kunci: Nasi
  -> Mengambil Halaman 1 untuk: 'Nasi'...
  -> Mengambil Halaman 2 untuk: 'Nasi'...
  -> Mengambil Halaman 3 untuk: 'Nasi'...
  -> Mengambil Halaman 4 untuk: 'Nasi'...
  -> Mengambil Halaman 5 untuk: 'Nasi'...
  -> Mengambil Halaman 6 untuk: 'Nasi'...
  -> Mengambil Halaman 7 untuk: 'Nasi'...
  -> Mengambil Halaman 8 untuk: 'Nasi'...
  -> Mengambil Halaman 9 untuk: 'Nasi'...
  -> Mengambil Halaman 10 untuk: 'Nasi'...
✅ Berhasil menyimpan total 59 item untuk kata kunci 'Nasi'

[2/42] Memulai pencarian kata kunci: Mie
  -> Mengambil Halaman 1 untuk: 'Mie'...
  -> Mengambil Halaman 2 untuk: 'Mie'...
  -> Mengambil Halaman 3 untuk: 'Mie'...
  -> Mengambil Halaman 4 untuk: 'Mie'...
  -> Mengambil Halaman 5 untuk: 'Mie'...
  -> Mengambil Halaman 6 untuk: 'Mie'...
  -> Mengambil Halaman 7 untuk: 'Mie'...
  -> 